# JAMS to Tablature Transcription Pipeline

This notebook implements the complete end-to-end algorithm to transform raw JAMS annotation files into structured JSON and human-readable ASCII guitar tablature.

### Pipeline Phases:
1. **Ingestion:** Load and normalize multiple JAMS namespaces (notes, chords, beats).
2. **Synchronization:** Align events from 6 separate string tracks into a unified timeline.
3. **Transcription:** Group simultaneous onsets and calculate fret positions.
4. **Rendering:** Output the finalized data in JSON and ASCII formats.
5. **Export:** Save the results to the `assets/transcriptions/` directory.

In [ ]:
import json
import numpy as np
import os
from collections import defaultdict

# Constants
JAMS_FILE = '../Annotations/00_BN1-129-Eb_comp.jams'
TIME_TOLERANCE = 0.05  # 50ms for grouping chords
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64] # Low E (0) to High E (5)
STRING_MAP = {0: 'string_6', 1: 'string_5', 2: 'string_4', 3: 'string_3', 4: 'string_2', 5: 'string_1'}

## 1. Data Normalization & Track Extraction

In [ ]:
def normalize_jams_data(data):
    if isinstance(data, list): return data
    times, durs, vals = data.get('time', []), data.get('duration', []), data.get('value', [])
    return [{'time': times[i], 'duration': durs[i], 'value': vals[i]} for i in range(len(times))]

with open(JAMS_FILE, 'r') as f:
    raw_jams = json.load(f)

# Extract Key Mode
key_anno = [a for a in raw_jams['annotations'] if a['namespace'] == 'key_mode']
key_signature = key_anno[0]['data'][0]['value'] if key_anno else "Unknown"

# Extract Chords
chord_anno = [a for a in raw_jams['annotations'] if a['namespace'] == 'chord']
chords = normalize_jams_data(chord_anno[0]['data']) if chord_anno else []

# Extract Beats
beat_anno = [a for a in raw_jams['annotations'] if a['namespace'] == 'beat_position']
beats = normalize_jams_data(beat_anno[0]['data']) if beat_anno else []

# Extract Note Tracks (GuitarSet has 6 note_midi tracks, one per string)
note_annos = [a for a in raw_jams['annotations'] if a['namespace'] == 'note_midi']
all_notes = []
for i, anno in enumerate(note_annos[:6]): # Ensure we only take the first 6 strings
    data = normalize_jams_data(anno['data'])
    for obs in data:
        all_notes.append({
            'time': obs['time'],
            'duration': obs['duration'],
            'midi': obs['value'],
            'string': i
        })

all_notes.sort(key=lambda x: x['time'])
print(f"Key: {key_signature}")
print(f"Total synced notes: {len(all_notes)}")

## 2. Transcription: Grouping & Fret Calculation

In [ ]:
def get_chord_at_time(t):
    for ch in chords:
        if ch['time'] <= t < (ch['time'] + ch['duration']): return ch['value']
    return ""

tab_segments = []
if all_notes:
    current_group = [all_notes[0]]

    for i in range(1, len(all_notes)):
        note = all_notes[i]
        if note['time'] - current_group[0]['time'] < TIME_TOLERANCE:
            current_group.append(note)
        else:
            # Process group into a segment
            start_time = current_group[0]['time']
            positions = {f"string_{s}": None for s in range(1, 7)}
            for n in current_group:
                fret = int(round(n['midi'] - OPEN_STRING_MIDI[n['string']]))
                positions[STRING_MAP[n['string']]] = fret

            tab_segments.append({
                "time_start": round(start_time, 3),
                "time_end": round(start_time + max(n['duration'] for n in current_group), 3),
                "suggested_chord": get_chord_at_time(start_time),
                "positions": positions
            })
            current_group = [note]

print(f"Generated {len(tab_segments)} tab segments (grouped chords/notes).")

## 3. Output Generation (JSON)

In [ ]:
output_json = {
    "key_signature": key_signature,
    "tab_segments": tab_segments
}

# Preview first 2 segments
print(json.dumps(output_json['tab_segments'][:2], indent=2))

## 4. ASCII Tablature Rendering Engine

In [ ]:
def generate_ascii_tab(segments, segments_per_line=4):
    string_keys = ["string_1", "string_2", "string_3", "string_4", "string_5", "string_6"]
    prefixes = ["e|", "B|", "G|", "D|", "A|", "E|"]
    col_width = 8
    full_tab = ""

    for i in range(0, len(segments), segments_per_line):
        chunk = segments[i:i + segments_per_line]

        # Chord line
        chord_line = "   "
        for seg in chunk:
            label = seg['suggested_chord'] if seg['suggested_chord'] else ""
            chord_line += label.ljust(col_width + 1)
        full_tab += chord_line.rstrip() + "\n"

        # String lines
        for s_idx, s_key in enumerate(string_keys):
            line = prefixes[s_idx]
            for seg in chunk:
                fret = seg['positions'][s_key]
                cell = str(fret) if fret is not None else "-"
                line += cell.center(col_width, "-") + "|"
            full_tab += line + "\n"

        full_tab += "\n"

    return full_tab

ascii_output = generate_ascii_tab(tab_segments)
print("ASCII Tab Generated (Length: {} characters)".format(len(ascii_output)))

## 5. Exporting Results
We save the results to the `assets/transcriptions/` directory with the requested naming convention.

In [ ]:
EXPORT_DIR = '../transcriptions'
if not os.path.exists(EXPORT_DIR):
    os.makedirs(EXPORT_DIR)

base_name = os.path.splitext(os.path.basename(JAMS_FILE))[0]
json_path = os.path.join(EXPORT_DIR, f"{base_name}_out_tab.json")
txt_path = os.path.join(EXPORT_DIR, f"{base_name}_out_tab.txt")

# Save JSON
with open(json_path, 'w') as f:
    json.dump(output_json, f, indent=2)

# Save ASCII
with open(txt_path, 'w') as f:
    f.write(ascii_output)

print(f"Successfully exported results to:\n - {json_path}\n - {txt_path}")